# 🪐 Planet Hunters — Exploratory Data Analysis & Signal Analysis
**ML Challenge Hack4Hack 2026 · Round 1**

This notebook provides astronomical signal exploration, light-curve morphology inspection, centroid vetting, and feature importance analysis across all 8 target classes.

In [ ]:
import os, sys
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns

# Ensure project modules can be loaded
sys.path.insert(0, str(Path.cwd().parent))
from src import config
from src.data_loader import load_stars_metadata, load_flux_arrays, load_centroid_arrays, load_cadences
from src.preprocessing import normalize_flux, process_centroids

## 1. Load Datasets and Inspect Class Distribution

In [ ]:
train_stars, test_stars = load_stars_metadata()
train_cad, test_cad = load_cadences()
train_flux, test_flux = load_flux_arrays()
train_cen, test_cen = load_centroid_arrays()

print(f"Train stars: {train_stars.shape}, Test stars: {test_stars.shape}")
print(f"Train flux:  {train_flux.shape}, Test flux:  {test_flux.shape}")
train_stars['label'].value_counts()

## 2. Visualize Archetypal Light Curves for All 8 Classes

In [ ]:
t = train_cad['time_btjd'].values
dumps = t[(train_cad['quality'].values & 1) > 0]

fig, axes = plt.subplots(len(config.CLASSES), 1, figsize=(14, 2.3 * len(config.CLASSES)), sharex=True)
for ax, lab in zip(axes, config.CLASSES):
    idx = train_stars.index[train_stars['label'] == lab][0]
    f = normalize_flux(train_flux[idx])
    ax.plot(t, f * 1e3, '.', ms=2, color='navy', alpha=0.7)
    for d in dumps:
        ax.axvline(d, color='orange', lw=0.6, alpha=0.5)
    ax.set_ylabel('ppt')
    ax.set_title(f"{lab.upper()} (Star ID: {train_stars.loc[idx, 'id']}, Tmag: {train_stars.loc[idx, 'tmag']:.1f})", fontsize=10, loc='left', fontweight='bold')
axes[-1].set_xlabel('Time (BTJD)', fontsize=11)
plt.tight_layout()
plt.show()

## 3. Centroid Vetting: Planets vs. Background Binaries
For background binaries, the light centroid shifts measurably during the eclipse because the blended background star dims. For genuine planetary transits, the centroid stays centered.

In [ ]:
planet_idx = train_stars.index[train_stars['label'] == 'planet'][0]
bg_idx = train_stars.index[train_stars['label'] == 'background_binary'][0]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
col_p, row_p, disp_p = process_centroids(train_cen[planet_idx])
col_bg, row_bg, disp_bg = process_centroids(train_cen[bg_idx])

ax1.plot(t, disp_p, '.', ms=2, color='green')
ax1.set_title(f"PLANET ({train_stars.loc[planet_idx, 'id']}) — Centroid Displacement is Stable", fontsize=10)
ax1.set_ylabel('Disp (millipixels)')

ax2.plot(t, disp_bg, '.', ms=2, color='crimson')
ax2.set_title(f"BACKGROUND BINARY ({train_stars.loc[bg_idx, 'id']}) — Notice Centroid Jumps During Dips", fontsize=10)
ax2.set_ylabel('Disp (millipixels)')
ax2.set_xlabel('Time (BTJD)')
plt.tight_layout()
plt.show()

## 4. Feature Importance of Best LightGBM Classifier

In [ ]:
import joblib
model_path = config.MODELS_DIR / 'best_lightgbm.joblib'
if model_path.exists():
    model = joblib.load(model_path)
    fi = pd.Series(model.feature_importances_, index=model.feature_name_).sort_values(ascending=False).head(20)
    plt.figure(figsize=(10, 6))
    fi.plot(kind='barh', color='teal')
    plt.gca().invert_yaxis()
    plt.title('Top 20 Most Predictive Features in LightGBM Model', fontsize=12, fontweight='bold')
    plt.xlabel('Split Count / Importance', fontsize=11)
    plt.tight_layout()
    plt.show()